# Single-pair d=3 probe: apparent interaction order in deployed detectors (Paper A)

Probe geometry of the theory: `X1, X2` independent standard Gaussian,
`corr(X1, X3) = rho` via `X3 = rho*X1 + sqrt(1-rho^2) Z`. Here the single-pair
closed-form law **does apply**:  F2(rho) = (1-rho^2)^2 / ((1+rho^2)(1+2rho^2)).

This experiment asks whether a **deployed** interaction detector reports an
interaction *order* that tracks the *identifiable* three-way content F2(rho).
Four function classes / methods, three of which return a holdout residual
variance fraction directly comparable to F2 = dist(h, S2)^2 / Var(h):

  * `frac_rbf_ridge`  -- RBF pairwise ridge = ADEQUATE class (byte-identical to
                         order_probe_v2). Estimates F2.
  * `frac_poly_D4`    -- degree-4 polynomial pairwise (byte-identical). ADEQUATE
                         for the monomial, IMPOVERISHED for the tanh product
                         (cannot represent tanh^2*tanh at any finite degree).
  * `frac_ebm`        -- ExplainableBoostingRegressor (purified GA2M, order-2 by
                         construction) = DEPLOYED adequate model. Estimates F2.
  * `nid_share3`      -- Neural Interaction Detection (Tsang et al. 2018) on an
                         MLP. Returns the 3-way interaction strength as a share
                         of the top pairwise strength -- a RANKING quantity, NOT
                         a variance fraction. It is RECORDED, not compared to F2
                         by subtraction, and never acceptance-checked.

PRE-REGISTERED, PROVABLE anchors are the only acceptance checks:
  monomial rho=0 -> 1 (independence); monomial rho=0.5 -> 0.30 (Theorem 2.9,
  closed form); monomial rho=1 -> 0 (degenerates to X1^2*X2, two-variable);
  tanh_prod rho=0 -> 1 (zero-mean product); tanh_prod rho=1 -> 0 (degenerates to
  tanh(X1)^2*tanh(X2)); and the pairwise control h = tanh(x1+x2)+tanh(x2+x3)+
  tanh(x1+x3), which lies in S2 by construction so F2 = 0 EXACTLY at every rho
  (checked on the RBF estimator).

The EMPIRICAL PHENOMENON is RECORDED, not tested (it is a finding, not a
theorem, and hard-coding it into a passing test would be circular):
  (i)   poly_D4 - RBF gap on the tanh product = basis-inflation of apparent order
        (large at high rho where the truth is ~0);
  (ii)  EBM ~ RBF on the tanh product = two adequate classes agree the 3-way
        content has collapsed;
  (iii) nid_share3 roughly flat in rho = the detector reports a strong 3-way
        interaction regardless of identifiability.

NOTE ON REPRODUCIBILITY: `frac_ebm` and `nid_share3` call sklearn / interpret and
are stochastic and library-version dependent -- unlike the pure-numpy projections,
their columns are NOT bitwise reproducible across machines. Library versions are
recorded in metadata.json. Outputs to
`MyDrive/KDD_Interactions/results/nid_ebm_order_probe/`.


In [ ]:
# Cell 1 -- Mount Drive and set up output folder
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/KDD_Interactions'
OUT = os.path.join(BASE, 'results', 'nid_ebm_order_probe')
os.makedirs(OUT, exist_ok=True)
print('output folder:', OUT)


In [ ]:
# Cell 2 -- Install the deployed EBM (purified GA2M). interpret-core is the
# lightweight subset that ships ExplainableBoostingRegressor.
!pip install -q interpret-core
try:
    from interpret.glassbox import ExplainableBoostingRegressor
    import interpret
    _HAVE_EBM = True
    print('EBM available; interpret', getattr(interpret, '__version__', 'unknown'))
except Exception as e:
    _HAVE_EBM = False
    print('EBM unavailable (frac_ebm will be NaN):', e)


In [ ]:
# Cell 3 -- Generator (single-pair) + estimators + provenance hash
import numpy as np, json, csv, time, hashlib, os, sklearn
from itertools import product as iproduct
from sklearn.neural_network import MLPRegressor

def make_data_singlepair(n, rho, seed, target):
    """Single correlated pair: X1,X2 independent; corr(X1,X3)=rho via
    X3 = rho*X1 + sqrt(1-rho^2) Z. This is the probe geometry of the theory,
    so the single-pair closed form F2(rho) applies."""
    # rho=1: X3 == X1 exactly (design singular in cols 0,2); intended --
    # the target then degenerates to a two-variable function.
    rng = np.random.default_rng(seed)
    x1 = rng.standard_normal(n)
    x2 = rng.standard_normal(n)
    z  = rng.standard_normal(n)
    x3 = rho * x1 + np.sqrt(max(0.0, 1.0 - rho ** 2)) * z
    X = np.column_stack([x1, x2, x3])
    if target == "monomial":
        h = x1 * x2 * x3
    elif target == "tanh_prod":
        h = np.tanh(x1) * np.tanh(x2) * np.tanh(x3)
    elif target == "pairwise_control":
        # lies in S_2 BY CONSTRUCTION: F_2 = 0 exactly at every rho.
        h = np.tanh(x1 + x2) + np.tanh(x2 + x3) + np.tanh(x1 + x3)
    else:
        raise ValueError(f"Unknown target: {target}")
    return X, h

# ---------- reused estimators: byte-identical logic to order_probe_v2 ----------
def monomial_exps(n_vars, D, max_active=2):
    out = []
    for combo in iproduct(range(D + 1), repeat=n_vars):
        if sum(combo) <= D and sum(1 for c in combo if c > 0) <= max_active:
            out.append(combo)
    return out

def frac_poly(X, h, D, max_active=2):
    exps = monomial_exps(3, D, max_active)
    cols = []
    for e in exps:
        col = np.ones(X.shape[0])
        for j, p in enumerate(e):
            if p > 0:
                col = col * X[:, j] ** p
        cols.append(col)
    Phi = np.column_stack(cols)
    mu = Phi.mean(0); sd = Phi.std(0); sd[sd == 0] = 1.0
    Phi = (Phi - mu) / sd
    const_idx = exps.index((0, 0, 0))
    Phi[:, const_idx] = 1.0
    hc = h - h.mean()
    denom = float(hc @ hc)
    if denom <= 0:
        return float("nan")
    beta, *_ = np.linalg.lstsq(Phi, hc, rcond=None)
    r = hc - Phi @ beta
    return float((r @ r) / denom)

CENTERS = np.linspace(-2.5, 2.5, 9)
BW = 0.75

def uni_feats(x):
    return np.column_stack([x] + [np.exp(-0.5 * ((x - c) / BW) ** 2) for c in CENTERS])

def design_rbf(X):
    n = X.shape[0]
    U = [uni_feats(X[:, j]) for j in range(3)]
    cols = [np.ones((n, 1))] + U
    for a, b in [(0, 1), (0, 2), (1, 2)]:
        cols.append((U[a][:, :, None] * U[b][:, None, :]).reshape(n, -1))
    return np.concatenate(cols, axis=1)

def frac_rbf_ridge(X, h, lam=10.0, split_seed=0, train_frac=0.75):
    """Holdout test-residual variance fraction under the RBF pairwise basis.
    Ridge does not penalize the intercept."""
    n = X.shape[0]
    idx = np.random.default_rng(split_seed).permutation(n)
    tr, te = idx[: int(train_frac * n)], idx[int(train_frac * n):]
    Phi = design_rbf(X)
    mu = Phi[tr].mean(0); sd = Phi[tr].std(0); sd[sd == 0] = 1.0
    Phi = (Phi - mu) / sd
    Phi[:, 0] = 1.0
    hm = h[tr].mean()
    P = np.eye(Phi.shape[1]); P[0, 0] = 0.0
    A = Phi[tr].T @ Phi[tr] + lam * P
    b = Phi[tr].T @ (h[tr] - hm)
    try:
        from scipy.linalg import cho_factor, cho_solve
        beta = cho_solve(cho_factor(A), b)
    except Exception:
        beta = np.linalg.solve(A, b)
    resid = (h[te] - hm) - Phi[te] @ beta
    denom = np.sum((h[te] - h[te].mean()) ** 2)
    if denom <= 0:
        return float("nan")
    return float((resid @ resid) / denom)

# ---------- NEW: deployed detector methods ----------
def frac_ebm(X, h, seed=0, train_frac=0.75):
    """Holdout residual variance fraction under a purified GA2M / EBM
    (order-2 by construction) -- a DEPLOYED estimate of dist(h, S2)^2 / Var(h).
    Stochastic and library-dependent; NOT bitwise reproducible."""
    if not _HAVE_EBM:
        return float("nan")
    n = X.shape[0]
    idx = np.random.default_rng(1000 + seed).permutation(n)
    tr, te = idx[: int(train_frac * n)], idx[int(train_frac * n):]
    m = ExplainableBoostingRegressor(interactions=3, max_bins=64,
                                     outer_bags=2, random_state=seed)
    m.fit(X[tr], h[tr])
    r = h[te] - m.predict(X[te])
    denom = np.sum((h[te] - h[te].mean()) ** 2)
    return float((r @ r) / denom) if denom > 0 else float("nan")

def nid_share3(X, h, seed=0):
    """Neural Interaction Detection (Tsang, Cheng, Liu 2018) on a 2-hidden-layer
    MLP. Interaction strength omega(I) = sum_k z_k * min_{i in I} |W0[i,k]|,
    with z = |W1| @ |W2| the aggregated later-layer influence. Returns the
    3-way strength as a share of the top pairwise strength -- a RANKING
    quantity, NOT a variance fraction. Stochastic; NOT bitwise reproducible."""
    m = MLPRegressor(hidden_layer_sizes=(64, 64), activation="relu", solver="adam",
                     alpha=1e-4, max_iter=400, early_stopping=True,
                     random_state=seed).fit(X, h)
    W0 = np.abs(m.coefs_[0])                              # (d, h1)
    z = (np.abs(m.coefs_[1]) @ np.abs(m.coefs_[2])).ravel()  # (h1,)
    om = lambda idx: float(np.sum(z * np.min(W0[idx, :], axis=0)))
    smax_pair = max(om([0, 1]), om([0, 2]), om([1, 2]))
    s123 = om([0, 1, 2])
    return s123 / smax_pair if smax_pair > 0 else float("nan")

def single_pair_F(rho):
    return (1 - rho ** 2) ** 2 / ((1 + rho ** 2) * (1 + 2 * rho ** 2))

N = 20_000
SEEDS = [0, 1, 2]
RHOS = [0.0, 0.3, 0.5, 0.7, 0.9, 0.99, 1.0]
TARGETS = ["monomial", "tanh_prod", "pairwise_control"]
DETECTOR_TARGETS = ["monomial", "tanh_prod"]   # NID + EBM only where a 3-way story exists
EBM_SEEDS = [0, 1]                             # EBM is slow; two seeds suffice
POLY_DEGS = [3, 4]
RIDGE_LAM = 10.0

_config = {"N": N, "SEEDS": SEEDS, "RHOS": RHOS, "TARGETS": TARGETS,
           "DETECTOR_TARGETS": DETECTOR_TARGETS, "EBM_SEEDS": EBM_SEEDS,
           "POLY_DEGS": POLY_DEGS, "RIDGE_LAM": RIDGE_LAM,
           "CENTERS": CENTERS.tolist(), "BW": BW, "train_frac": 0.75,
           "geometry": "single_pair", "mlp": [64, 64], "ebm_bags": 2}
CODE_SHA = hashlib.sha256(
    b"".join(f.__code__.co_code for f in
             [make_data_singlepair, monomial_exps, frac_poly, uni_feats,
              design_rbf, frac_rbf_ridge, frac_ebm, nid_share3])
    + json.dumps(_config, sort_keys=True).encode()).hexdigest()
print("provenance sha256 (bytecode + config):", CODE_SHA)
print("note: frac_ebm / nid_share3 are stochastic + library-dependent and are")
print("      NOT covered by bitwise reproducibility (see metadata for versions).")


In [ ]:
# Cell 4 -- Sweep, results to Drive
EXP = "nid_ebm_order_probe"
t0 = time.time()
rows = []
for target in TARGETS:
    for rho in RHOS:
        for s in SEEDS:
            X, h = make_data_singlepair(N, rho, s, target)
            rec = {"experiment": EXP, "target": target, "rho": rho, "seed": s}
            for D in POLY_DEGS:
                rec[f"frac_poly_D{D}"] = frac_poly(X, h, D)
            rec["frac_rbf_ridge"] = frac_rbf_ridge(X, h, lam=RIDGE_LAM, split_seed=s)
            if target in DETECTOR_TARGETS:
                rec["nid_share3"] = nid_share3(X, h, seed=s)
                rec["frac_ebm"] = frac_ebm(X, h, seed=s) if s in EBM_SEEDS else float("nan")
            else:
                rec["nid_share3"] = float("nan")
                rec["frac_ebm"] = float("nan")
            rows.append(rec)
        print(f"{target:16s} rho={rho:4.2f} done  ({time.time()-t0:6.1f}s)", flush=True)

FIELDS = ["experiment", "target", "rho", "seed"] + \
         [f"frac_poly_D{D}" for D in POLY_DEGS] + ["frac_rbf_ridge", "frac_ebm", "nid_share3"]
with open(os.path.join(OUT, "per_seed.csv"), "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=FIELDS)
    w.writeheader(); w.writerows(rows)

import collections
summ = collections.defaultdict(list)
for r in rows:
    summ[(r["target"], r["rho"])].append(r)
NUMCOLS = [f"frac_poly_D{D}" for D in POLY_DEGS] + ["frac_rbf_ridge", "frac_ebm", "nid_share3"]
sum_rows = []
for (target, rho), rs in summ.items():
    rec = {"experiment": EXP, "target": target, "rho": rho, "n_seeds": len(rs)}
    for col in NUMCOLS:
        vals = [r[col] for r in rs]
        rec[col + "_mean"] = float(np.nanmean(vals))
        rec[col + "_sd"] = float(np.nanstd(vals))
    sum_rows.append(rec)
with open(os.path.join(OUT, "results.csv"), "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(sum_rows[0].keys()))
    w.writeheader(); w.writerows(sum_rows)

try:
    import interpret as _itp; _itp_ver = getattr(_itp, "__version__", "unknown")
except Exception:
    _itp_ver = "unavailable"
with open(os.path.join(OUT, "metadata.json"), "w") as f:
    json.dump({"experiment": EXP, "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
               "geometry": "single_pair: corr(X1,X3)=rho, X2 independent",
               "N": N, "seeds": SEEDS, "rhos": RHOS, "targets": TARGETS,
               "detector_targets": DETECTOR_TARGETS, "ebm_seeds": EBM_SEEDS,
               "poly_degs": POLY_DEGS, "ridge_lam": RIDGE_LAM,
               "code_sha256": CODE_SHA, "numpy": np.__version__,
               "sklearn": sklearn.__version__, "interpret": _itp_ver,
               "reproducibility_note": "frac_ebm and nid_share3 are stochastic "
               "and library-dependent; not bitwise reproducible across versions."},
              f, indent=2)
print("wrote per_seed.csv, results.csv, metadata.json")


In [ ]:
# Cell 5 -- Verification from disk: PROVABLE anchors acceptance-checked;
# the empirical phenomenon (NID inflation, cross-class gap) RECORDED, not tested.
# Anchor bounds are calibrated for N>=20000; they tighten toward the paper's
# N=60000 values as N grows (poly rho=0 -> 0.995, RBF rho=0 -> 1.01, rho=0.5 -> 0.303).
import csv as _csv
rows = list(_csv.DictReader(open(os.path.join(OUT, "per_seed.csv"))))
assert all(r["experiment"] == "nid_ebm_order_probe" for r in rows), "experiment stamp mismatch (stale file?)"

def agg(t, rho, col):
    v = [float(r[col]) for r in rows
         if r["target"] == t and abs(float(r["rho"]) - rho) < 1e-9 and r[col] != "" and r[col] != "nan"]
    v = [x for x in v if not np.isnan(x)]
    return (float(np.mean(v)), float(np.std(v))) if v else (float("nan"), float("nan"))

law = lambda r: (1 - r**2) ** 2 / ((1 + r**2) * (1 + 2 * r**2))

print(f"{'target':16s}{'rho':>5s}  {'poly D4':>14s}  {'RBF (adeq)':>14s}  {'EBM (adeq)':>14s}  {'NID 3-share':>12s}  {'F2 law':>9s}")
for t in TARGETS:
    for rho in RHOS:
        m4, s4 = agg(t, rho, "frac_poly_D4")
        mr, _  = agg(t, rho, "frac_rbf_ridge")
        me, _  = agg(t, rho, "frac_ebm")
        mn, _  = agg(t, rho, "nid_share3")
        ref = f"{law(rho):9.4f}" if t in ("monomial", "tanh_prod") else "       --"
        print(f"{t:16s}{rho:5.2f}  {m4:6.4f}+-{s4:6.4f}  {mr:14.4f}  {me:14.4f}  {mn:12.4f}  {ref}")
    print()

checks = []
m,_ = agg("monomial", 0.0, "frac_poly_D4"); checks.append(("monomial rho=0 -> ~1 (independence anchor, provable)", 0.96 < m < 1.01))
m,_ = agg("monomial", 0.5, "frac_poly_D4"); checks.append(("monomial rho=0.5 -> ~0.30 (Theorem 2.9 closed form)", 0.27 < m < 0.34))
m,_ = agg("monomial", 1.0, "frac_poly_D4"); checks.append(("monomial rho=1 -> ~0 (degenerates to X1^2*X2)", m < 0.02))
m,_ = agg("tanh_prod", 0.0, "frac_rbf_ridge"); checks.append(("tanh_prod rho=0 -> ~1 (zero-mean product anchor)", 0.95 < m < 1.10))
m,_ = agg("tanh_prod", 1.0, "frac_rbf_ridge"); checks.append(("tanh_prod rho=1 -> ~0 (degenerates to tanh(X1)^2*tanh(X2))", m < 0.02))
pc_ok = all(agg("pairwise_control", r, "frac_rbf_ridge")[0] < 0.02 for r in RHOS)
checks.append(("pairwise_control in S_2 by construction: RBF fraction < 0.02 at all rho", pc_ok))

story = []
for name, ok in checks:
    line = ("PASS  " if ok else "FAIL  ") + name
    story.append(line); print(line)

# ---- RECORDED observations (the empirical phenomenon; NOT acceptance tests) ----
gap  = [agg("tanh_prod", r, "frac_poly_D4")[0] - agg("tanh_prod", r, "frac_rbf_ridge")[0] for r in RHOS]
ebmr = [agg("tanh_prod", r, "frac_ebm")[0] for r in RHOS]
rbfr = [agg("tanh_prod", r, "frac_rbf_ridge")[0] for r in RHOS]
nid_m = [agg("monomial", r, "nid_share3")[0] for r in RHOS]
nid_t = [agg("tanh_prod", r, "nid_share3")[0] for r in RHOS]
story.append("OBS   basis-inflation gap (tanh_prod, poly_D4 - RBF), by rho " + str(RHOS) + ": " + " ".join(f"{v:.3f}" for v in gap))
story.append("OBS   adequate-class agreement (tanh_prod): RBF " + " ".join(f"{v:.3f}" for v in rbfr))
story.append("OBS                                         EBM " + " ".join(f"{v:.3f}" for v in ebmr))
story.append("OBS   NID 3-way share (monomial, ~flat in rho): " + " ".join(f"{v:.3f}" for v in nid_m))
story.append("OBS   NID 3-way share (tanh_prod, ~flat in rho): " + " ".join(f"{v:.3f}" for v in nid_t))
story.append("OBS   F2 law reference, by rho: " + " ".join(f"{law(r):.3f}" for r in RHOS))
for s in story[len(checks):]:
    print(s)

with open(os.path.join(OUT, "check.txt"), "w") as f:
    f.write("\n".join(story) + "\n")
print("\nwrote check.txt")
